### Clustering | onsets included | dmode separate | foot-pelvis

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

today = datetime.now().strftime("%d%b").lower()

with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

m_idx = 1
mode = ["group", "individual", "audience"]
dance_mode = mode[m_idx]

cluster_data = {
    "file_name": [],    # string
    
    "dmode_name": [],   # string
    "dmode_seg_idx": [], # int
    "dmode_start": [],  # float
    "dmode_end": [],    # float
    
    "cycle_idx": [],    # int
    "cycle_start": [],  # float
    "cycle_end": [],    # float
    
    "location": [],     # string
    "ensemble": [],     # string
    "day": [],          # string
    "rec_no": [], # string
    "piece": [],        # string
    
    "L_traj": [],      # numpy array
    "R_traj": [],       # numpy array
    "LR_traj": [],       # numpy array
    "pelvis_traj": [],
    
    "L_onsets": [],
    "R_onsets": [],
    
    "Dun": [],
    "J1": [],
    "J2": [],
}

# left and right foot onsets by cycle per mode
# three drum onsets by cycle per mode



motion_data_dir = "data/motion_data_pkl"
base_path_cycles = "data/virtual_cycles"
base_path_logs= "data/logs_v4_0.007_foot_jun3"           # "data/logs_v2_may",   # "data/logs_v3_0.2_lower_jun3",    # "data/logs_v4_0.007_foot_jun3", 



for file_name in piece_list:
    
    # print(file_name)
    # file_name = piece_list[0]  # BKO_E1_D1_01_Suku
    location = file_name.split("_")[0]      # BKO
    ensemble = file_name.split("_")[1]      # E1
    day = file_name.split("_")[2]           # D1
    recording_no = file_name.split("_")[3]  # 01
    piece = file_name.split("_")[4]         # Suku


    onsets_csv_path = f"data/drum_onsets/{file_name}.csv"
    dance_mode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"
    drum_onsets_path = f"data/drum_onsets/{file_name}.csv"

    # build file paths
    cycles_csv = os.path.join(base_path_cycles, f"{file_name}_C.csv")
    logs_onset_dir = os.path.join(base_path_logs, f"{file_name}_T", "onset_info")

    left_onsets_csv  = os.path.join(logs_onset_dir, f"{file_name}_T_left_foot_onsets.csv")
    right_onsets_csv = os.path.join(logs_onset_dir, f"{file_name}_T_right_foot_onsets.csv")
    left_zpos_csv    = os.path.join(logs_onset_dir, f"{file_name}_T_left_foot_zpos.csv")
    right_zpos_csv   = os.path.join(logs_onset_dir, f"{file_name}_T_right_foot_zpos.csv")
    
    # Load the mocap pickle file
    mpkl_path = f"{motion_data_dir}/{file_name}_T.pkl"
    with open(mpkl_path, 'rb') as f:
        motion_data = pickle.load(f)

    # pelvis position data
    pelvis_zpos = motion_data["position"]['SEGMENT_PELVIS'][:,2]        # mocap 240 fps
    pelvis_n_frames = len(pelvis_zpos)
    pelvis_times = np.arange(pelvis_n_frames) / 240
    
    # Load feet onset data
    L_onsets = pd.read_csv(left_onsets_csv)["time_sec"].values
    R_onsets = pd.read_csv(right_onsets_csv)["time_sec"].values
    
    Dun_onsets = pd.read_csv(drum_onsets_path)["Dun"].values
    J1_onsets = pd.read_csv(drum_onsets_path)["J1"].values
    J2_onsets = pd.read_csv(drum_onsets_path)["J2"].values

    
    # load feet position data
    Left_zpos = pd.read_csv(left_zpos_csv)["zpos"].values
    Right_zpos = pd.read_csv(right_zpos_csv)["zpos"].values
    n_frames = len(Left_zpos)
    times = np.arange(n_frames) / 240

    # load cycles
    cyc_df = pd.read_csv(cycles_csv)

    # load dance mode time segments
    if os.path.exists(dance_mode_path):
        with open(dance_mode_path, "rb") as f:
            dmode_ts = pickle.load(f)
    else:
        print(f"{file_name} {dance_mode} does not exist")
        continue
    # print(location, ensemble, day, recording_no, piece)
    # print(dance_mode)
    # print(dmode_ts) 
    
    
    for dmode_idx, dmode in enumerate(dmode_ts):
        dmode_start, dmode_end = dmode
        # print("dmode_seg_no:", dmode_idx+1)
        # print("dmode_segment:", dmode_start, dmode_end)

        onsets = cyc_df["Virtual Onset"].values     # in seconds

        # Filter onsets to get only those within the dance mode time segment
        mode_mask = (onsets >= dmode_start) & (onsets <= dmode_end)
        mode_onsets = onsets[mode_mask]

        # Create list of tuples with start and end times of cycles within the dance mode segment
        cycle_times = [(round(mode_onsets[i], 3), round(mode_onsets[i+1], 3)) for i in range(len(mode_onsets)-1)]
        # print("cycle_times:", cycle_times)

        for c_idx, (c_start, c_end) in enumerate(cycle_times):
            # print("cycle:", c_idx+1, c_start, c_end)
        
            win_mask = (times >= c_start) & (times <= c_end)
            t_win = times[win_mask]
            
            # feet
            Left_zpos_win = Left_zpos[win_mask]
            Right_zpos_win = Right_zpos[win_mask]
            left_right_zpos = np.concatenate((Left_zpos_win, Right_zpos_win))
            
            # pelvis
            pelvis_win_mask = (pelvis_times >= c_start) & (pelvis_times <= c_end)
            pelvis_t_win = pelvis_times[pelvis_win_mask]
            pelvis_zpos_win = pelvis_zpos[pelvis_win_mask]
            
            # feet onsets
            c_left_onset = L_onsets[(L_onsets >= c_start) & (L_onsets <= c_end)]
            c_right_onset = R_onsets[(R_onsets >= c_start) & (R_onsets <= c_end)]
            # drum onsets
            c_Dun_onset = Dun_onsets[(Dun_onsets >= c_start) & (Dun_onsets <= c_end)]
            c_J1_onset = J1_onsets[(J1_onsets >= c_start) & (J1_onsets <= c_end)]
            c_J2_onset = J2_onsets[(J2_onsets >= c_start) & (J2_onsets <= c_end)]
            
            # save 
            cluster_data["file_name"].append(file_name)
            
            cluster_data["dmode_name"].append(dance_mode)
            cluster_data["dmode_seg_idx"].append(dmode_idx+1)
            cluster_data["dmode_start"].append(dmode_start)
            cluster_data["dmode_end"].append(dmode_end)
            
            cluster_data["cycle_idx"].append(c_idx+1)
            cluster_data["cycle_start"].append(c_start)
            cluster_data["cycle_end"].append(c_end)

            cluster_data["location"].append(location)
            cluster_data["ensemble"].append(ensemble)
            cluster_data["day"].append(day)
            cluster_data["rec_no"].append(recording_no)
            cluster_data["piece"].append(piece)
            
            cluster_data["L_traj"].append(Left_zpos_win)
            cluster_data["R_traj"].append(Right_zpos_win)
            cluster_data["LR_traj"].append(left_right_zpos)
            cluster_data["pelvis_traj"].append(pelvis_zpos_win)
            
            cluster_data["L_onsets"].append(c_left_onset)
            cluster_data["R_onsets"].append(c_right_onset)
            
            cluster_data["Dun"].append(c_Dun_onset)
            cluster_data["J1"].append(c_J1_onset)
            cluster_data["J2"].append(c_J2_onset)
            
        
            # plt.plot(pelvis_t_win, pelvis_zpos_win, label='Left Foot')
            # plt.legend()
            # plt.show()

out_dir = os.path.join("data", "cluster_data", "mvnx")
os.makedirs(out_dir, exist_ok=True)

save_pkl = os.path.join(out_dir, f"cluster_data_mvnx_{dance_mode}_{today}.pkl")
save_mat = os.path.join(out_dir, f"cluster_data_mvnx_{dance_mode}_with_pelvis_{today}.mat")

with open(save_pkl, "wb") as f:
    pickle.dump(cluster_data, f)

from scipy.io import savemat
savemat(save_mat, cluster_data)


### Clustering | onsets included | dmode separate | hand-foot-pelvis

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
today = datetime.now().strftime("%d%b").lower()

with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

m_idx = 1
mode = ["group", "individual", "audience"]
dance_mode = mode[m_idx]

cluster_data = {
    "file_name": [],    # string
    
    "dmode_name": [],   # string
    "dmode_seg_idx": [], # int
    "dmode_start": [],  # float
    "dmode_end": [],    # float
    
    "cycle_idx": [],    # int
    "cycle_start": [],  # float
    "cycle_end": [],    # float
    
    "location": [],     # string
    "ensemble": [],     # string
    "day": [],          # string
    "rec_no": [], # string
    "piece": [],        # string
    
    "L_hand_ztraj": [],      # numpy array
    "R_hand_ztraj": [],       # numpy array
    "LR_hand_ztraj": [],       # numpy array
    
    "L_foot_ztraj": [],      # numpy array
    "R_foot_ztraj": [],       # numpy array
    "LR_foot_ztraj": [],       # numpy array
    
    "pelvis_ztraj": [],
    
    "L_hand_zonsets": [],
    "R_hand_zonsets": [],
    
    "both_foot_zonsets": [],
    "L_foot_zonsets": [],
    "R_foot_zonsets": [],
    
    "Dun": [],
    "J1": [],
    "J2": [],
}

# left and right foot onsets by cycle per mode
# three drum onsets by cycle per mode


mocap_fps = 240
motion_data_dir = "data/motion_data_pkl"

for file_name in piece_list:
    
    # print(file_name)
    # file_name = piece_list[0]  # BKO_E1_D1_01_Suku
    location = file_name.split("_")[0]      # BKO
    ensemble = file_name.split("_")[1]      # E1
    day = file_name.split("_")[2]           # D1
    recording_no = file_name.split("_")[3]  # 01
    piece = file_name.split("_")[4]         # Suku


    base_path_cycles = "data/virtual_cycles"
    # base_path_logs= "data/logs_v4_0.007_foot_jun3"           # "data/logs_v2_may",   # "data/logs_v3_0.2_lower_jun3",    # "data/logs_v4_0.007_foot_jun3", 
    dance_mode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"
    drum_onsets_path = f"data/drum_onsets/{file_name}.csv"

    # build file paths
    cycles_csv = os.path.join(base_path_cycles, f"{file_name}_C.csv")
    feet_onset_dir = os.path.join("data/logs_v4_0.007_foot_jun3", f"{file_name}_T", "onset_info")
    hand_onset_dir = os.path.join("data/output_hand_onsets_12nov25", f"{file_name}_T", "onset_info")


    left_foot_onsets_csv  = os.path.join(feet_onset_dir, f"{file_name}_T_left_foot_onsets.csv")
    right_foot_onsets_csv = os.path.join(feet_onset_dir, f"{file_name}_T_right_foot_onsets.csv")
    
    left_hand_onsets_csv  = os.path.join(hand_onset_dir, f"{file_name}_T_left_hand_onsets.csv")
    right_hand_onsets_csv = os.path.join(hand_onset_dir, f"{file_name}_T_right_hand_onsets.csv")
    
    
    # Load the mocap pickle file
    mpkl_path = f"{motion_data_dir}/{file_name}_T.pkl"
    with open(mpkl_path, 'rb') as f:
        motion_data = pickle.load(f)

    # pelvis position data
    pelvis_zpos = motion_data["segments"]['Pelvis']['position'][:,2]        # mocap 240 fps
    pelvis_n_frames = len(pelvis_zpos)
    pelvis_times = np.arange(pelvis_n_frames) / mocap_fps
    
    # load feet position data
    Left_foot_zpos = motion_data["segments"]['LeftFoot']['position'][:,2]
    Right_foot_zpos = motion_data["segments"]['RightFoot']['position'][:,2]
    
    Left_hand_zpos = motion_data["segments"]['LeftHand']['position'][:,2]
    Right_hand_zpos = motion_data["segments"]['RightHand']['position'][:,2]
    
    
    n_frames = len(Left_foot_zpos)
    times = np.arange(n_frames) / mocap_fps
    
    # Load feet onset data
    L_foot_onsets = pd.read_csv(left_foot_onsets_csv)["time_sec"].values
    R_foot_onsets = pd.read_csv(right_foot_onsets_csv)["time_sec"].values
    
    # Load hand onset data
    L_hand_onsets = pd.read_csv(left_hand_onsets_csv)["time_sec"].values
    R_hand_onsets = pd.read_csv(right_hand_onsets_csv)["time_sec"].values
    
    # Load drum onset data
    Dun_onsets = pd.read_csv(drum_onsets_path)["Dun"].values
    J1_onsets = pd.read_csv(drum_onsets_path)["J1"].values
    J2_onsets = pd.read_csv(drum_onsets_path)["J2"].values

    # load cycles
    cyc_df = pd.read_csv(cycles_csv)

    # load dance mode time segments
    if os.path.exists(dance_mode_path):
        with open(dance_mode_path, "rb") as f:
            dmode_ts = pickle.load(f)
    else:
        print(f"{file_name} {dance_mode} does not exist")
        continue
    # print(location, ensemble, day, recording_no, piece)
    # print(dance_mode)
    # print(dmode_ts) 
    
    
    for dmode_idx, dmode in enumerate(dmode_ts):
        dmode_start, dmode_end = dmode
        # print("dmode_seg_no:", dmode_idx+1)
        # print("dmode_segment:", dmode_start, dmode_end)

        onsets = cyc_df["Virtual Onset"].values     # in seconds

        # Filter onsets to get only those within the dance mode time segment
        mode_mask = (onsets >= dmode_start) & (onsets <= dmode_end)
        mode_onsets = onsets[mode_mask]

        # Create list of tuples with start and end times of cycles within the dance mode segment
        cycle_times = [(round(mode_onsets[i], 3), round(mode_onsets[i+1], 3)) for i in range(len(mode_onsets)-1)]
        # print("cycle_times:", cycle_times)

        for c_idx, (c_start, c_end) in enumerate(cycle_times):
            # print("cycle:", c_idx+1, c_start, c_end)

            win_mask = (times >= c_start) & (times <= c_end)
            t_win = times[win_mask]
            
            # position trajectories ----------------------------------
            
            #hand
            left_hand_zpos_win = Left_hand_zpos[win_mask]
            right_hand_zpos_win = Right_hand_zpos[win_mask]
            left_right_hand_zpos = np.concatenate((left_hand_zpos_win, right_hand_zpos_win))
            
            # feet
            left_foot_zpos_win = Left_foot_zpos[win_mask]
            right_foot_zpos_win = Right_foot_zpos[win_mask]
            left_right_feet_zpos = np.concatenate((left_foot_zpos_win, right_foot_zpos_win))
            
            # pelvis
            pelvis_win_mask = (pelvis_times >= c_start) & (pelvis_times <= c_end)
            pelvis_t_win = pelvis_times[pelvis_win_mask]
            pelvis_zpos_win = pelvis_zpos[pelvis_win_mask]
            
            # Onsets -------------------------------------------------
            # feet onsets
            c_left_feet_zonset = L_foot_onsets[(L_foot_onsets >= c_start) & (L_foot_onsets <= c_end)]
            c_right_feet_zonset = R_foot_onsets[(R_foot_onsets >= c_start) & (R_foot_onsets <= c_end)]
            
            # hand onsets
            c_left_hand_zonset = L_hand_onsets[(L_hand_onsets >= c_start) & (L_hand_onsets <= c_end)]
            c_right_hand_zonset = R_hand_onsets[(R_hand_onsets >= c_start) & (R_hand_onsets <= c_end)]
            
            # drum onsets
            c_Dun_onset = Dun_onsets[(Dun_onsets >= c_start) & (Dun_onsets <= c_end)]
            c_J1_onset = J1_onsets[(J1_onsets >= c_start) & (J1_onsets <= c_end)]
            c_J2_onset = J2_onsets[(J2_onsets >= c_start) & (J2_onsets <= c_end)]
            
            # save 
            cluster_data["file_name"].append(file_name)
            
            cluster_data["dmode_name"].append(dance_mode)
            cluster_data["dmode_seg_idx"].append(dmode_idx+1)
            cluster_data["dmode_start"].append(dmode_start)
            cluster_data["dmode_end"].append(dmode_end)
            
            cluster_data["cycle_idx"].append(c_idx+1)
            cluster_data["cycle_start"].append(c_start)
            cluster_data["cycle_end"].append(c_end)

            cluster_data["location"].append(location)
            cluster_data["ensemble"].append(ensemble)
            cluster_data["day"].append(day)
            cluster_data["rec_no"].append(recording_no)
            cluster_data["piece"].append(piece)
            
            cluster_data["L_hand_ztraj"].append(left_hand_zpos_win)
            cluster_data["R_hand_ztraj"].append(right_hand_zpos_win)
            cluster_data["LR_hand_ztraj"].append(left_right_hand_zpos)
            
            cluster_data["L_foot_ztraj"].append(left_foot_zpos_win)
            cluster_data["R_foot_ztraj"].append(right_foot_zpos_win)
            cluster_data["LR_foot_ztraj"].append(left_right_feet_zpos)
            
            cluster_data["pelvis_ztraj"].append(pelvis_zpos_win)
            
            cluster_data["L_hand_zonsets"].append(c_left_hand_zonset)
            cluster_data["R_hand_zonsets"].append(c_right_hand_zonset)           
            
            cluster_data["L_foot_zonsets"].append(c_left_feet_zonset)
            cluster_data["R_foot_zonsets"].append(c_right_feet_zonset)
            
            cluster_data["Dun"].append(c_Dun_onset)
            cluster_data["J1"].append(c_J1_onset)
            cluster_data["J2"].append(c_J2_onset)
            
        
            # plt.plot(pelvis_t_win, pelvis_zpos_win, label='Left Foot')
            # plt.legend()
            # plt.show()

# with open(f"cluster_data_{dance_mode}_{today}.pkl", "wb") as f:
#     pickle.dump(cluster_data, f)

# from scipy.io import savemat
# savemat(f"cluster_data_{dance_mode}_hand_feet_pelvis_{today}.mat", cluster_data)


### Clustering | onsets included | ALL dmodes combined | hand-foot-pelvis


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
today = datetime.now().strftime("%d%b").lower()

with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

# Process ALL dance modes instead of just one
modes = ["group", "individual", "audience"]

cluster_data = {
    "file_name": [],    # string
    
    "dmode_name": [],   # string
    "dmode_seg_idx": [], # int
    "dmode_start": [],  # float
    "dmode_end": [],    # float
    
    "cycle_idx": [],    # int
    "cycle_start": [],  # float
    "cycle_end": [],    # float
    
    "location": [],     # string
    "ensemble": [],     # string
    "day": [],          # string
    "rec_no": [], # string
    "piece": [],        # string
    
    "L_hand_zPos": [],      # numpy array
    "R_hand_zPos": [],       # numpy array
    "LR_hand_zPos": [],       # numpy array
    
    "L_foot_zPos": [],      # numpy array
    "R_foot_zPos": [],       # numpy array
    "LR_foot_zPos": [],       # numpy array
    
    "pelvis_zPos": [],
    
    "L_hand_zonsets": [],
    "R_hand_zonsets": [],
    
    "L_foot_zonsets": [],
    "R_foot_zonsets": [],
    
    "Dun": [],
    "J1": [],
    "J2": [],
}

# left and right foot onsets by cycle per mode
# three drum onsets by cycle per mode

mocap_fps = 240
motion_data_dir = "data/motion_data_pkl"

for file_name in piece_list:
    
    # print(file_name)
    # file_name = piece_list[0]  # BKO_E1_D1_01_Suku
    location = file_name.split("_")[0]      # BKO
    ensemble = file_name.split("_")[1]      # E1
    day = file_name.split("_")[2]           # D1
    recording_no = file_name.split("_")[3]  # 01
    piece = file_name.split("_")[4]         # Suku

    base_path_cycles = "data/virtual_cycles"
    drum_onsets_path = f"data/drum_onsets/{file_name}.csv"

    # build file paths
    cycles_csv = os.path.join(base_path_cycles, f"{file_name}_C.csv")
    
    onset_dir = os.path.join("data/output-nov13", f"{file_name}_T")
    feet_onset_path = os.path.join(onset_dir, f"{file_name}_T_feet_onset.pkl")
    hand_onset_path = os.path.join(onset_dir, f"{file_name}_T_hands_onset.pkl")
    
    with open(feet_onset_path, 'rb') as f:
        feet_onset_data = pickle.load(f)
    
    with open(hand_onset_path, 'rb') as f:
        hand_onset_data = pickle.load(f)
    
    # Load the mocap pickle file
    mpkl_path = f"{motion_data_dir}/{file_name}_T.pkl"
    with open(mpkl_path, 'rb') as f:
        motion_data = pickle.load(f)

    # pelvis position data
    pelvis_zpos = motion_data["segments"]['Pelvis']['position'][:,2]        # mocap 240 fps
    pelvis_n_frames = len(pelvis_zpos)
    pelvis_times = np.arange(pelvis_n_frames) / mocap_fps
    
    # load feet position data
    Left_foot_zpos = motion_data["segments"]['LeftFoot']['position'][:,2]
    Right_foot_zpos = motion_data["segments"]['RightFoot']['position'][:,2]
    
    Left_hand_zpos = motion_data["segments"]['LeftHand']['position'][:,2]
    Right_hand_zpos = motion_data["segments"]['RightHand']['position'][:,2]
    
    n_frames = len(Left_foot_zpos)
    times = np.arange(n_frames) / mocap_fps
    
    # Load feet onset data
    L_foot_onsets = feet_onset_data["left_foot_zOnsets"]["time_sec"]
    R_foot_onsets = feet_onset_data["right_foot_zOnsets"]["time_sec"]
    
    # Load hand onset data  
    L_hand_onsets = hand_onset_data["left_hand"]["time_sec"]
    R_hand_onsets = hand_onset_data["right_hand"]["time_sec"]
    
    # Load drum onset data
    Dun_onsets = pd.read_csv(drum_onsets_path)["Dun"].values
    J1_onsets = pd.read_csv(drum_onsets_path)["J1"].values
    J2_onsets = pd.read_csv(drum_onsets_path)["J2"].values

    # load cycles
    cyc_df = pd.read_csv(cycles_csv)

    # Loop over ALL dance modes for this file
    for dance_mode in modes:
        dance_mode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"
        
        # load dance mode time segments
        if os.path.exists(dance_mode_path):
            with open(dance_mode_path, "rb") as f:
                dmode_ts = pickle.load(f)
        else:
            print(f"{file_name} {dance_mode} does not exist")
            continue
        
        for dmode_idx, dmode in enumerate(dmode_ts):
            dmode_start, dmode_end = dmode
            # print("dmode_seg_no:", dmode_idx+1)
            # print("dmode_segment:", dmode_start, dmode_end)

            onsets = cyc_df["Virtual Onset"].values     # in seconds

            # Filter onsets to get only those within the dance mode time segment
            mode_mask = (onsets >= dmode_start) & (onsets <= dmode_end)
            mode_onsets = onsets[mode_mask]

            # Create list of tuples with start and end times of cycles within the dance mode segment
            cycle_times = [(round(mode_onsets[i], 3), round(mode_onsets[i+1], 3)) for i in range(len(mode_onsets)-1)]
            # print("cycle_times:", cycle_times)

            for c_idx, (c_start, c_end) in enumerate(cycle_times):
                # print("cycle:", c_idx+1, c_start, c_end)

                win_mask = (times >= c_start) & (times <= c_end)
                t_win = times[win_mask]
                
                # position trajectories ----------------------------------
                
                #hand
                left_hand_zpos_win = Left_hand_zpos[win_mask]
                right_hand_zpos_win = Right_hand_zpos[win_mask]
                left_right_hand_zpos = np.concatenate((left_hand_zpos_win, right_hand_zpos_win))
                
                # feet
                left_foot_zpos_win = Left_foot_zpos[win_mask]
                right_foot_zpos_win = Right_foot_zpos[win_mask]
                left_right_feet_zpos = np.concatenate((left_foot_zpos_win, right_foot_zpos_win))
                
                # pelvis
                pelvis_win_mask = (pelvis_times >= c_start) & (pelvis_times <= c_end)
                pelvis_t_win = pelvis_times[pelvis_win_mask]
                pelvis_zpos_win = pelvis_zpos[pelvis_win_mask]
                
                # Onsets -------------------------------------------------
                # feet onsets
                c_left_feet_zonset = L_foot_onsets[(L_foot_onsets >= c_start) & (L_foot_onsets <= c_end)]
                c_right_feet_zonset = R_foot_onsets[(R_foot_onsets >= c_start) & (R_foot_onsets <= c_end)]
                
                # hand onsets
                c_left_hand_zonset = L_hand_onsets[(L_hand_onsets >= c_start) & (L_hand_onsets <= c_end)]
                c_right_hand_zonset = R_hand_onsets[(R_hand_onsets >= c_start) & (R_hand_onsets <= c_end)]
                
                # drum onsets
                c_Dun_onset = Dun_onsets[(Dun_onsets >= c_start) & (Dun_onsets <= c_end)]
                c_J1_onset = J1_onsets[(J1_onsets >= c_start) & (J1_onsets <= c_end)]
                c_J2_onset = J2_onsets[(J2_onsets >= c_start) & (J2_onsets <= c_end)]
                
                # save 
                cluster_data["file_name"].append(file_name)
                
                cluster_data["dmode_name"].append(dance_mode)
                cluster_data["dmode_seg_idx"].append(dmode_idx+1)
                cluster_data["dmode_start"].append(dmode_start)
                cluster_data["dmode_end"].append(dmode_end)
                
                cluster_data["cycle_idx"].append(c_idx+1)
                cluster_data["cycle_start"].append(c_start)
                cluster_data["cycle_end"].append(c_end)

                cluster_data["location"].append(location)
                cluster_data["ensemble"].append(ensemble)
                cluster_data["day"].append(day)
                cluster_data["rec_no"].append(recording_no)
                cluster_data["piece"].append(piece)
                
                cluster_data["L_hand_zPos"].append(left_hand_zpos_win)
                cluster_data["R_hand_zPos"].append(right_hand_zpos_win)
                cluster_data["LR_hand_zPos"].append(left_right_hand_zpos)
                
                cluster_data["L_foot_zPos"].append(left_foot_zpos_win)
                cluster_data["R_foot_zPos"].append(right_foot_zpos_win)
                cluster_data["LR_foot_zPos"].append(left_right_feet_zpos)
                
                cluster_data["pelvis_zPos"].append(pelvis_zpos_win)
                
                cluster_data["L_hand_zonsets"].append(c_left_hand_zonset)
                cluster_data["R_hand_zonsets"].append(c_right_hand_zonset)           
                
                cluster_data["L_foot_zonsets"].append(c_left_feet_zonset)
                cluster_data["R_foot_zonsets"].append(c_right_feet_zonset)
                
                cluster_data["Dun"].append(c_Dun_onset)
                cluster_data["J1"].append(c_J1_onset)
                cluster_data["J2"].append(c_J2_onset)
            
        
                # plt.plot(pelvis_t_win, pelvis_zpos_win, label='Left Foot')
                # plt.legend()
                # plt.show()

out_dir = os.path.join("data", "cluster_data", "mvnx")
os.makedirs(out_dir, exist_ok=True)

save_pkl = os.path.join(out_dir, f"cluster_data_mvnx_all_dmodes_hand_feet_pelvis_with_onsets_{today}.pkl")
save_mat = os.path.join(out_dir, f"cluster_data_mvnx_all_dmodes_hand_feet_pelvis_with_onsets_{today}.mat")

with open(save_pkl, "wb") as f:
    pickle.dump(cluster_data, f)

from scipy.io import savemat
savemat(save_mat, cluster_data)


**Clustering data extraction Nov-13 Latest**

Extracts position, velocity, acceleration, magnitudes, and center of mass data for all body parts from motion capture data, organized by dance mode and cycle.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.io import savemat
from clustering import process_all_files, initialize_cluster_data

today = datetime.now().strftime("%d%b").lower()

# Load piece list
with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

# Process ALL dance modes
modes = ["group", "individual", "audience"]

print(f"Processing {len(piece_list)} files...")
print(f"Dance modes: {modes}")

# Process all files
cluster_data = process_all_files(
    piece_list=piece_list,
    modes=modes,
    mocap_fps=240,
    motion_data_dir="data/motion_data_pkl",
    base_path_cycles="data/virtual_cycles",
    debug=False  # Set to False to reduce output
)

# Save the cluster data in a dedicated MVNX subdirectory
out_dir = os.path.join("data", "cluster_data", "mvnx")
os.makedirs(out_dir, exist_ok=True)

save_pkl = os.path.join(out_dir, f"cluster_data_mvnx_all_body_parts_{today}.pkl")
save_mat = os.path.join(out_dir, f"cluster_data_mvnx_all_body_parts_{today}.mat")

if cluster_data is not None:
    with open(save_pkl, 'wb') as f:
        pickle.dump(cluster_data, f)
    print(f"[INFO] Saved cluster data to: {save_pkl}")

    print("Saving to MAT file...")
    savemat(save_mat, cluster_data)
    
    print(f"[INFO] Saved cluster data to: {save_mat}")
    
    # Print summary statistics
    print(f"[INFO] Summary:")
    print(f"  Total cycles: {len(cluster_data['file_name'])}")
    print(f"  Body parts: {list(cluster_data['body_parts'].keys())}")
    print(f"  Files processed: {len(set(cluster_data['file_name']))}")
    print(f"  Dance modes: {set(cluster_data['dmode_name'])}")
else:
    print("[ERROR] Failed to process files")

## Build 7 x 3 x 2 x 101 cluster representation (real-time world coords -> cycle-phase grid)

Builds, **per cycle**, a `(7, 3, 2, 101)` array from the Nov-13 `cluster_data` above:

- **7 clusters** (index order): `COM, RL, LL, RH, LH, body, Head`
  - `COM` = `center_of_mass` **as-is**
  - `RL/LL` = mean of `Upper/Lower-Leg + Foot` (per side)
  - `RH/LH` = mean of `UpperArm + ForeArm + Hand` (per side)
  - `body` = mean of `Pelvis, T8`
  - `Head` = `Head` alone
- **3** = world `x, y, z` (meters, not phase-normalized)
- **2** = `[position=0, velocity=1]`
- **101** = cycle-phase grid `X = arange(0, 1.01, 0.01)`; `C(T) = (t - cycle_start)/(cycle_end - cycle_start)`, linear interp `F(C) -> F(X)`

Does **not** modify the original extraction. Saves to `data/cluster_data/cluster_7x3x2_<today>.pkl` and `.mat`.

In [ ]:
import pickle
from scipy.io import loadmat
from clustering import process_all_files
from cluster_7x3x2_mvnx import build_cluster_7x3x2, save_cluster_7x3x2, CLUSTER_MEMBERS

# Reuse the in-memory Nov-13 cluster_data if it exists; otherwise rebuild it.
# cluster_data = loadmat('data/cluster_data/cluster_data_all_body_parts_15dec.mat')

try:
    cluster_data
    print("[INFO] Reusing in-memory cluster_data from the Nov-13 cell.")
except NameError:
    print("[INFO] cluster_data not in memory; rebuilding via process_all_files ...")
    with open('data/selected_piece_list.pkl', 'rb') as f:
        piece_list = pickle.load(f)
    cluster_data = process_all_files(
        piece_list=piece_list,
        modes=["group", "individual", "audience"],
        mocap_fps=240,
        motion_data_dir="data/motion_data_pkl",
        base_path_cycles="data/virtual_cycles",
        debug=False,
    )

print("[INFO] Cluster membership:")
for name, members in CLUSTER_MEMBERS.items():
    print(f"   {name:5s} <- {'center_of_mass (as-is)' if members is None else members}")

# Step 1-2 (world-coord pos/vel per cluster) + Step 3 (resample to phase grid).
out = build_cluster_7x3x2(cluster_data, phase_step=0.01)

print(f"[INFO] data shape: {out['data'].shape}  (n_cycles, cluster, xyz, [pos,vel], phase)")
print(f"[INFO] phase grid: {out['phase_grid'][0]:.2f} .. {out['phase_grid'][-1]:.2f} "
      f"({len(out['phase_grid'])} points)")

out_dir = "data/cluster_data/mvnx"
paths = save_cluster_7x3x2(out, out_dir=out_dir)  # add save_npy=True for a raw .npy
print("[INFO] Saved:", paths)